# 03 · Distill the OneK1K teacher on Geuvadis

[Previous: OneK1K training](02_onek1k_reference_training.ipynb) · [Tutorial index](../README.md#try-it-out)

Now we'll apply the **memory B cell** teacher to each Geuvadis individual's
personalized blood embeddings, then fit interpretable cis-genotype students.
Set `CELL_TYPE` in the first code cell to the same label you chose in notebook 02.
Prediction and distillation use that teacher's single cell type automatically.
The students learn its predictions, so no measured Geuvadis expression is needed.

The default is **probabilistic, heterozygote-aware distillation**. You can also
choose deterministic or purely additive variants below. This step produces the
student models and covariance files that bundled MetaXcan will use in notebook 04.

Taking the shortcut? Download the [pretrained OneK1K linear models](https://huggingface.co/Padraig7/atticus-onek1k) to `models/onek1k/` and head straight to [notebook 04](04_twas.ipynb).

In [ ]:
CELL_TYPE = "memory B cell"  # Use the same label in notebooks 02–04.

from pathlib import Path
import sys

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "pyproject.toml").is_file()
             and (path / "src/training/run.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or its notebooks/ directory.")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))
from tutorial import ROOT, paths_for_cell_type, run, uv, require, student_draw_count, load_native_checkpoint

EXECUTE = False
WANDB_ARGS = ["--wandb-mode", "disabled"]

PATHS = paths_for_cell_type(CELL_TYPE)


## Choose a distillation variant

| `MODE` | `ADDITIVE` | What gets fitted |
| --- | --- | --- |
| `"probabilistic"` | `False` | Default: dosage plus heterozygote indicators, 5 teacher members × 5 bootstrap fits. |
| `"probabilistic"` | `True` | Dosage-only model with the same member/bootstrap draws. |
| `"deterministic"` | `False` | One dosage-plus-heterozygote fit to the ensemble-mean target. |
| `"deterministic"` | `True` | One dosage-only fit to the ensemble-mean target. |

Probabilistic fits use aleatoric precision weights and bootstrap samples of the
training individuals. Deterministic fits use neither bootstrapping nor precision
weighting. From the command line, `--additive` selects the dosage-only version.


In [ ]:
MODE = "probabilistic"
ADDITIVE = False
JOBS = 1

if MODE not in {"probabilistic", "deterministic"}:
    raise ValueError("MODE must be 'probabilistic' or 'deterministic'.")
STUDENT_DIR = PATHS["output"] / f"students-{MODE}-{'additive' if ADDITIVE else 'het'}"
STUDENTS = STUDENT_DIR / "students.h5"
print(f"Student output: {STUDENT_DIR}")


## Check personalized inputs

Each individual in the sample list needs a complete archive at
`data/embeddings/geuvadis/blood/<sample>.npz`. Genotype files use one naming template:
`chr{chrom}.GRCh38.biallelic.{bed,bim,fam}`.

These are Geuvadis individuals. The [OneK1K embeddings on Hugging Face](https://huggingface.co/datasets/Padraig7/atticus-onek1k) belong to a different cohort; prepare this panel's embeddings in [notebook 01](01_data_and_embeddings.ipynb).

The next cell counts the expected archives and checks the first archive's
metadata. During prediction, every archive is checked for its sample ID, tissue,
gene order, feature dimensions, and values. The notebook displays counts rather
than individual identifiers or expression values.


In [ ]:
import json
import h5py
import numpy as np
import pandas as pd
from IPython.display import display
from src.preprocessing.geuvadis import read_ids
from src.distillation.genotypes import TEMPLATE
from src.distillation.run import split_samples

sample_ids = read_ids(PATHS["samples"]) if PATHS["samples"].is_file() else []
embedding_files = [PATHS["personalized"] / f"{sample}.npz" for sample in sample_ids]
missing_embeddings = [path for path in embedding_files if not path.is_file()]
print(f"Teacher checkpoint exists: {PATHS['teacher'].is_file()}")
if PATHS["teacher"].is_file():
    checkpoint = load_native_checkpoint(PATHS["teacher"])
    if checkpoint["data"]["cell_types"] != [CELL_TYPE]:
        raise ValueError("Run notebook 02 with the same CELL_TYPE before distillation.")
    del checkpoint
print(f"Reference individuals: {len(sample_ids)}")
print(f"Personalized archives present: {len(embedding_files) - len(missing_embeddings)}/{len(embedding_files)}")

chromosomes = [str(chrom) for chrom in range(1, 23)]
if embedding_files and embedding_files[0].is_file():
    with np.load(embedding_files[0], allow_pickle=False) as archive:
        expected = {"features", "ensids", "chroms", "tss", "sample_id", "tissue"}
        if not expected <= set(archive.files):
            raise ValueError(f"Personalized archive is missing fields: {sorted(expected - set(archive.files))}")
        if str(archive["sample_id"].item()) != sample_ids[0] or str(archive["tissue"].item()) != "blood":
            raise ValueError("Expected the selected individual's blood embeddings.")
        chromosomes = sorted({str(chrom).lower().removeprefix("chr") for chrom in archive["chroms"]}, key=int)
        print(f"Genes in first archive: {len(archive['ensids']):,}; chromosomes: {len(chromosomes)}")

genotype_files = [PATHS["genotypes"] / f"{TEMPLATE.format(chrom=chrom)}.{extension}"
                  for chrom in chromosomes for extension in ("bed", "bim", "fam")]
print(f"PLINK files present: {sum(path.is_file() for path in genotype_files)}/{len(genotype_files)}")
if len(sample_ids) >= 5:
    train_indices, test_indices = split_samples(sample_ids, seed=42)
    print(f"Distillation individuals: {len(train_indices)} training, {len(test_indices)} held out")
if not sample_ids or missing_embeddings:
    print("Finish Geuvadis preparation and personalized embeddings in notebook 01 before execution.")


## Export the selected cell type's five teacher members

We'll reuse the OneK1K checkpoint to save means and aleatoric variances with axes
**gene × member × individual × cell type**. Member indices are local to each
cell type's independent ensemble.


In [ ]:
prediction_command = uv(
    "distillation", "src/distillation/predict.py",
    "--teacher", PATHS["teacher"],
    "--embeddings", PATHS["personalized"],
    "--samples", PATHS["samples"],
    "-o", PATHS["predictions"],
    "--batch-size", 256,
    "--device", "auto",
    *WANDB_ARGS,
)
if EXECUTE:
    require([PATHS["teacher"], PATHS["samples"], *embedding_files])
run(prediction_command, execute=EXECUTE)


In [ ]:
if PATHS["predictions"].is_file():
    with h5py.File(PATHS["predictions"], "r") as predictions:
        if predictions["cell_types"].asstr()[:].tolist() != [CELL_TYPE]:
            raise ValueError("Prediction cell type does not match CELL_TYPE.")
        print(f"Complete: {bool(predictions.attrs.get('complete', False))}")
        print(f"Prediction scale: {predictions.attrs['target_scale']}")
        display(pd.DataFrame({"axis": ["gene", "teacher member", "individual", "OneK1K cell type"],
                              "size": predictions["means"].shape}))
else:
    print("Prediction metadata will appear after the export completes.")


## Fit students and save their covariances

Distillation uses a fixed **80/20 individual split**. The standard
503-person panel gives 402 training and 101 held-out individuals.

Every cis SNP that passes genotype QC enters the fit. By default, that means a
complete, biallelic SNP within **±500 kb of the TSS**, with reference-panel
**MAF ≥ 0.05**.

After **each gene/cell-type fit**, `students.h5` saves covariance upper triangles
for the nonzero projected weights. Covariance uses **all reference individuals**,
including held-out individuals. The final PredictDB and covariance files are then
assembled under `models/`, one pair per cell type and draw, ready for MetaXcan.


In [ ]:
distillation_command = uv(
    "distillation", "src/distillation/run.py",
    "-y", PATHS["predictions"],
    "--genotypes", PATHS["genotypes"],
    "-o", STUDENT_DIR,
    "--mode", MODE,
    *(["--additive"] if ADDITIVE else []),
    "--half-window", 500_000,
    "--maf", 0.05,
    "--bootstraps", 5,
    "--l1-ratio", 0.5,
    "--cv", 3,
    "--alphas", 15,
    "--seed", 42,
    "--split-seed", 42,
    "--jobs", JOBS,
    *WANDB_ARGS,
)
if EXECUTE:
    require([PATHS["predictions"], *genotype_files])
run(distillation_command, execute=EXECUTE)


Again, for optional local W&B logs, set `WANDB_ARGS` in the first code cell to:

```python
WANDB_ARGS = ["--wandb-project", "atticus", "--wandb-group", "onek1k-tutorial",
              "--wandb-mode", "offline"]
```

Choose `online` after setting up your account for remote logging.


## Inspect fidelity and sparsity

The table reads at most three fitted genes for your selected cell type, so a
quick look stays quick even for large results. Spearman correlation measures **student
fidelity to teacher predictions** on held-out distillation individuals. It doesn't
measure accuracy against observed expression.


In [ ]:
expected_exports = None
complete = covariances_complete = False
if STUDENTS.is_file():
    rows = []
    with h5py.File(STUDENTS, "r") as students:
        config = json.loads(students["config"].asstr()[()])
        cells = students["cell_types"].asstr()[:]
        if cells.tolist() != [CELL_TYPE]:
            raise ValueError("Student cell type does not match CELL_TYPE.")
        complete = bool(students.attrs.get("complete", False))
        covariances_complete = bool(students.attrs.get("covariances_complete", False))
        print(f"Students complete: {complete}; covariances complete: {covariances_complete}")
        print(f"Individuals: {len(students['train_indices'])} training, {len(students['test_indices'])} held out")
        print(f"Feature screening: {config['feature_screening']}")
        draw_count = student_draw_count(students)
        if draw_count is not None:
            expected_exports = len(cells) * draw_count
        inspected_genes = 0
        reference_reported = False
        for gene, group in students["genes"].items():
            gene_has_fit = False
            for cell, cell_type in enumerate(cells):
                student = group.get(str(cell))
                if student is None or not student.attrs.get("fit_complete", group.attrs.get("complete", False)):
                    continue
                evaluation = json.loads(student.attrs["evaluation"])
                nonzeros = {
                    label: float(np.count_nonzero(np.abs(student[key][:]) > 1e-12, axis=-1).mean())
                    for label, key in (("nonzero dosage", "dosage_draws"),
                                       ("nonzero heterozygote", "heterozygote_draws"),
                                       ("nonzero projected", "coefficient_draws"))
                }
                rows.append({"gene": gene, "cell_type": cell_type,
                             "Spearman full": evaluation["test_full"].get("spearman"),
                             "Spearman projected": evaluation["test_additive_projection"].get("spearman"),
                             **nonzeros})
                gene_has_fit = True
                ld = student.get("ld")
                if not reference_reported and ld is not None and ld.attrs.get("complete", False):
                    print(f"Covariance reference individuals: {ld.attrs['n_samples']}; ddof: {ld.attrs['ddof']}")
                    reference_reported = True
            inspected_genes += int(gene_has_fit)
            if inspected_genes == 3:
                break
    if rows:
        display(pd.DataFrame(rows))
    else:
        print("No completed fits to display yet; genes without variable QC-passing cis SNPs are skipped.")
else:
    print("Student diagnostics will appear after distillation produces students.h5.")


In [ ]:
model_directory = STUDENT_DIR / "models"
markers = list(model_directory.glob("cell_*/complete.json"))
ready_draws = sum(len(json.loads(path.read_text())["draws"]) for path in markers)
export_config_path = model_directory / "config.json"
if expected_exports is None and export_config_path.is_file():
    export_config = json.loads(export_config_path.read_text())
    expected_exports = len(export_config["cell_types"]) * int(np.prod(export_config["draw_shape"]))
print(f"Cell types with completed exports: {len(markers)}")
print(f"Exported model/covariance pairs: {ready_draws}")
if expected_exports is not None:
    print(f"Expected pairs when complete: {expected_exports}")
if EXECUTE and (not complete or not covariances_complete or ready_draws != expected_exports):
    raise RuntimeError("Distillation or MetaXcan exports are incomplete; rerun the same command to resume.")


The default output directory is `outputs/tutorial/onek1k/memory_B_cell/students-probabilistic-het/`:

- `students.h5`: fits, held-out fidelity, coefficient draws, and stored covariances.
- `run.json`: input fingerprints and scientific settings.
- `models/cell_XXXX/member_M_bootstrap_B.db`: projected weights in PredictDB format.
- Matching `.cov.txt.gz` files: complete covariance upper triangles for each draw.

A student with zero weights is a valid result of regularization. It can't produce
a TWAS association, and the next step reports these cases.

**Next:** [04 · Run TWAS on the bundled GWAS files](04_twas.ipynb).
